# Embeddings -- Foundations

> **Section 01** -- Topic 02 -- `foundations`

**Prerequisites:** `01-tokenization` (all notebooks)

**What you'll learn:**
- Why embeddings are the foundation of all modern NLP
- How Word2Vec (Skip-gram and CBOW) and GloVe work internally
- How to evaluate embedding quality

**What you'll build:**
- Word2Vec (Skip-gram and CBOW) from scratch in PyTorch
- GloVe from scratch
- An analogy solver and embedding evaluation toolkit

---
## Setup

In [ ]:
# !pip install torch numpy matplotlib scikit-learn

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from collections import Counter, defaultdict
import random
import math

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

---
## 1. Why Embeddings? From One-Hot to Dense

Before embeddings, the standard way to represent words in a machine learning model was **one-hot encoding**. In this scheme, every word in the vocabulary gets its own dimension, and a word is represented as a vector of all zeros except for a single 1 at the position corresponding to that word's index.

This approach has several critical problems. First, the vectors are enormous -- if your vocabulary has 50,000 words, every word is a 50,000-dimensional vector with only a single non-zero entry. This is extremely wasteful in terms of memory and computation. Second, and more fundamentally, one-hot vectors encode **no semantic information whatsoever**. The cosine similarity between any two one-hot vectors is exactly zero, meaning the representation treats "cat" and "kitten" as no more related than "cat" and "democracy."

The **distributional hypothesis** -- the idea that "you shall know a word by the company it keeps" (Firth, 1957) -- provides the theoretical foundation for dense embeddings. Words that appear in similar contexts should have similar meanings, and therefore similar representations. This insight led to the development of Word2Vec, GloVe, and ultimately the contextual embeddings used in modern transformers.

In [ ]:
# Demonstrate one-hot encoding and its limitations

vocabulary = ["king", "queen", "man", "woman", "prince", "princess",
              "cat", "dog", "fish", "democracy"]
word_to_idx = {word: i for i, word in enumerate(vocabulary)}
vocab_size = len(vocabulary)


def one_hot_encode(word: str) -> np.ndarray:
    """Create a one-hot vector for a word."""
    vec = np.zeros(vocab_size)
    vec[word_to_idx[word]] = 1.0
    return vec


# Encode some words
king_vec = one_hot_encode("king")
queen_vec = one_hot_encode("queen")
cat_vec = one_hot_encode("cat")

print(f"Vocabulary size: {vocab_size}")
print(f"\n'king'  one-hot: {king_vec}")
print(f"'queen' one-hot: {queen_vec}")
print(f"'cat'   one-hot: {cat_vec}")

In [ ]:
# Show the sparsity and similarity problems

def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    """Compute cosine similarity between two vectors."""
    dot = np.dot(a, b)
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot / (norm_a * norm_b)


# Every pair of one-hot vectors has zero similarity
print("Cosine similarities between one-hot vectors:")
pairs = [("king", "queen"), ("king", "man"), ("king", "cat"), ("cat", "dog")]
for w1, w2 in pairs:
    sim = cosine_similarity(one_hot_encode(w1), one_hot_encode(w2))
    print(f"  sim('{w1}', '{w2}') = {sim:.4f}")

print(f"\nSparsity: {(vocab_size - 1) / vocab_size * 100:.1f}% zeros in each vector")
print(f"Memory for 50k vocab: {50_000 * 50_000 * 4 / 1e9:.1f} GB (float32 matrix)")
print("\n--> One-hot encoding captures ZERO semantic relationships!")

### The Solution: Dense Embeddings

Instead of using sparse, high-dimensional one-hot vectors, we want to learn **dense, low-dimensional** vectors (typically 50-300 dimensions) where semantic relationships are captured by the geometry of the space. Words with similar meanings should be close together, and ideally, meaningful directions in the space should correspond to semantic properties.

The key insight is that we can learn these vectors from data by training a model on a simple prediction task. The weights learned by the model become our word embeddings. This is the foundation of Word2Vec, which we'll implement next.

---
## 2. Word2Vec -- Skip-gram

The **Skip-gram** model, introduced by Mikolov et al. (2013), learns word embeddings by training a simple neural network on the following task: given a center word, predict the surrounding context words. The architecture is surprisingly simple -- just an embedding layer (input) and a linear layer (output) -- but the learned embeddings capture rich semantic relationships.

The training objective maximizes the probability of observing the actual context words given the center word. For a center word $w_c$ and context word $w_o$, the basic softmax probability is:

$$P(w_o | w_c) = \frac{\exp(\mathbf{u}_{w_o}^\top \mathbf{v}_{w_c})}{\sum_{w=1}^{V} \exp(\mathbf{u}_w^\top \mathbf{v}_{w_c})}$$

where $\mathbf{v}_{w_c}$ is the center word embedding and $\mathbf{u}_{w_o}$ is the context word embedding. Computing the full softmax over the entire vocabulary is prohibitively expensive, so in practice we use **negative sampling**: instead of normalizing over all words, we train the model to distinguish the true context word from a handful of randomly sampled "negative" words.

With negative sampling, the objective for a single (center, context) pair becomes:

$$\mathcal{L} = \log \sigma(\mathbf{u}_{w_o}^\top \mathbf{v}_{w_c}) + \sum_{i=1}^{k} \mathbb{E}_{w_i \sim P_n(w)} \left[ \log \sigma(-\mathbf{u}_{w_i}^\top \mathbf{v}_{w_c}) \right]$$

where $k$ is the number of negative samples and $P_n(w)$ is the noise distribution (typically the unigram distribution raised to the 3/4 power).

In [ ]:
# Training corpus -- a small but structured corpus for demonstration
# In practice you'd use millions of sentences; here we repeat patterns
# so the model can learn meaningful relationships.

corpus_sentences = [
    "the king ruled the kingdom with wisdom and power",
    "the queen ruled the kingdom with grace and wisdom",
    "the prince will become the next king of the kingdom",
    "the princess will become the next queen of the kingdom",
    "the man walked to the castle with the woman",
    "the woman walked to the castle with the man",
    "the king and the queen lived in the royal castle",
    "the prince and the princess played in the royal garden",
    "the boy and the girl went to the royal school",
    "the man and the woman worked in the city",
    "a dog chased a cat through the garden",
    "the cat climbed the tree while the dog barked",
    "a small dog played with a big cat in the park",
    "the fish swam in the river near the castle",
    "the bird flew over the kingdom and the city",
    "the king was a wise man who loved the queen",
    "the queen was a wise woman who loved the king",
    "the prince was a young boy in the kingdom",
    "the princess was a young girl in the kingdom",
    "the man and the boy walked the dog in the park",
    "the woman and the girl petted the cat in the garden",
    "the royal family lived in the grand castle",
    "the king gave power to the prince and the princess",
    "the queen gave wisdom to the prince and the princess",
    "the dog and the cat became friends in the park",
    "the fish jumped out of the river near the garden",
    "a wise king and a wise queen ruled the land",
    "the boy wanted to become a king someday",
    "the girl wanted to become a queen someday",
    "the man was strong and the woman was strong",
]

# Repeat corpus to give more training signal
corpus_sentences = corpus_sentences * 10

# Tokenize (simple whitespace tokenization for this demo)
corpus_tokens = []
for sentence in corpus_sentences:
    corpus_tokens.extend(sentence.lower().split())

# Build vocabulary
word_counts = Counter(corpus_tokens)
vocab = sorted(word_counts.keys())
word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for w, i in word2idx.items()}
V = len(vocab)

print(f"Corpus size: {len(corpus_tokens)} tokens")
print(f"Vocabulary size: {V} unique words")
print(f"Most common: {word_counts.most_common(10)}")

In [ ]:
# Create training pairs for Skip-gram: (center_word, context_word)

def create_skipgram_pairs(tokens, word2idx, window_size=2):
    """Generate (center, context) index pairs from a token list."""
    pairs = []
    for i, center in enumerate(tokens):
        center_idx = word2idx[center]
        # Look at words within the window on both sides
        for j in range(max(0, i - window_size), min(len(tokens), i + window_size + 1)):
            if i == j:
                continue
            context_idx = word2idx[tokens[j]]
            pairs.append((center_idx, context_idx))
    return pairs


WINDOW_SIZE = 3
skipgram_pairs = create_skipgram_pairs(corpus_tokens, word2idx, window_size=WINDOW_SIZE)
print(f"Generated {len(skipgram_pairs)} skip-gram training pairs")
print(f"\nExamples (first 5):")
for center_idx, context_idx in skipgram_pairs[:5]:
    print(f"  center='{idx2word[center_idx]}' -> context='{idx2word[context_idx]}'")

In [ ]:
# Build the negative sampling distribution
# P(w) proportional to count(w)^(3/4), as in the original paper

def build_noise_distribution(word_counts, word2idx, power=0.75):
    """Build unigram distribution raised to 3/4 power for negative sampling."""
    freqs = np.zeros(len(word2idx))
    for word, idx in word2idx.items():
        freqs[idx] = word_counts[word] ** power
    freqs /= freqs.sum()
    return freqs


noise_dist = build_noise_distribution(word_counts, word2idx)
print("Noise distribution (first 5 words):")
for i in range(5):
    print(f"  '{idx2word[i]}': {noise_dist[i]:.4f}")

In [ ]:
class SkipGramNegSampling(nn.Module):
    """Skip-gram model with negative sampling.

    The model maintains two embedding matrices:
    - center_embeddings: embeddings for when a word appears as the center word
    - context_embeddings: embeddings for when a word appears as context

    The final word vectors are typically taken from center_embeddings,
    though some implementations average both.
    """

    def __init__(self, vocab_size: int, embedding_dim: int):
        super().__init__()
        self.center_embeddings = nn.Embedding(vocab_size, embedding_dim)
        self.context_embeddings = nn.Embedding(vocab_size, embedding_dim)

        # Initialize with small random values (as in original implementation)
        initrange = 0.5 / embedding_dim
        self.center_embeddings.weight.data.uniform_(-initrange, initrange)
        self.context_embeddings.weight.data.uniform_(-initrange, initrange)

    def forward(self, center_ids, context_ids, negative_ids):
        """
        Args:
            center_ids:   (batch_size,) center word indices
            context_ids:  (batch_size,) positive context word indices
            negative_ids: (batch_size, num_negatives) negative sample indices

        Returns:
            Negative sampling loss (scalar)
        """
        # Get embeddings: (batch_size, embed_dim)
        center_vecs = self.center_embeddings(center_ids)
        context_vecs = self.context_embeddings(context_ids)
        neg_vecs = self.context_embeddings(negative_ids)  # (batch, num_neg, embed_dim)

        # Positive score: dot product between center and true context
        pos_score = torch.sum(center_vecs * context_vecs, dim=1)  # (batch,)
        pos_loss = torch.nn.functional.logsigmoid(pos_score)      # (batch,)

        # Negative scores: dot product between center and each negative sample
        # center_vecs: (batch, embed_dim) -> (batch, embed_dim, 1)
        neg_score = torch.bmm(neg_vecs, center_vecs.unsqueeze(2)).squeeze(2)  # (batch, num_neg)
        neg_loss = torch.nn.functional.logsigmoid(-neg_score).sum(dim=1)      # (batch,)

        # Total loss (we negate because we want to maximize)
        loss = -(pos_loss + neg_loss).mean()
        return loss

    def get_embeddings(self):
        """Return the center word embeddings as a numpy array."""
        return self.center_embeddings.weight.data.cpu().numpy()


print("SkipGramNegSampling model defined.")

In [ ]:
# Training the Skip-gram model

EMBEDDING_DIM = 50
NUM_NEGATIVES = 5
BATCH_SIZE = 256
NUM_EPOCHS = 50
LEARNING_RATE = 0.01

model_sg = SkipGramNegSampling(V, EMBEDDING_DIM).to(device)
optimizer_sg = optim.Adam(model_sg.parameters(), lr=LEARNING_RATE)

# Convert pairs to tensors
pairs_array = np.array(skipgram_pairs)
center_indices = torch.tensor(pairs_array[:, 0], dtype=torch.long, device=device)
context_indices = torch.tensor(pairs_array[:, 1], dtype=torch.long, device=device)

losses = []
num_batches = math.ceil(len(skipgram_pairs) / BATCH_SIZE)

for epoch in range(NUM_EPOCHS):
    # Shuffle training data
    perm = torch.randperm(len(skipgram_pairs), device=device)
    center_shuf = center_indices[perm]
    context_shuf = context_indices[perm]

    epoch_loss = 0.0
    for i in range(0, len(skipgram_pairs), BATCH_SIZE):
        batch_center = center_shuf[i:i + BATCH_SIZE]
        batch_context = context_shuf[i:i + BATCH_SIZE]
        batch_size = batch_center.size(0)

        # Sample negatives from noise distribution
        neg_samples = torch.tensor(
            np.random.choice(V, size=(batch_size, NUM_NEGATIVES), p=noise_dist),
            dtype=torch.long, device=device
        )

        optimizer_sg.zero_grad()
        loss = model_sg(batch_center, batch_context, neg_samples)
        loss.backward()
        optimizer_sg.step()

        epoch_loss += loss.item()

    avg_loss = epoch_loss / num_batches
    losses.append(avg_loss)
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch + 1:3d}/{NUM_EPOCHS} | Loss: {avg_loss:.4f}")

print("\nSkip-gram training complete!")

In [ ]:
# Plot training loss
plt.figure(figsize=(8, 4))
plt.plot(losses, linewidth=2)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Skip-gram Training Loss")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Visualize learned embeddings with t-SNE

def plot_embeddings(embeddings, idx2word, title="Word Embeddings (t-SNE)", words_to_plot=None):
    """Reduce embeddings to 2D with t-SNE and plot."""
    if words_to_plot is None:
        words_to_plot = list(idx2word.values())

    indices = [word2idx[w] for w in words_to_plot if w in word2idx]
    vecs = embeddings[indices]
    labels = [idx2word[i] for i in indices]

    # t-SNE reduction
    perplexity = min(5, len(indices) - 1)
    tsne = TSNE(n_components=2, random_state=42, perplexity=perplexity)
    vecs_2d = tsne.fit_transform(vecs)

    plt.figure(figsize=(10, 8))
    plt.scatter(vecs_2d[:, 0], vecs_2d[:, 1], alpha=0.6, s=80)
    for i, label in enumerate(labels):
        plt.annotate(label, (vecs_2d[i, 0], vecs_2d[i, 1]),
                     fontsize=11, fontweight="bold",
                     xytext=(5, 5), textcoords="offset points")
    plt.title(title, fontsize=14)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


# Plot selected semantically interesting words
interesting_words = ["king", "queen", "prince", "princess", "man", "woman",
                     "boy", "girl", "dog", "cat", "fish", "castle", "kingdom"]

sg_embeddings = model_sg.get_embeddings()
plot_embeddings(sg_embeddings, idx2word, title="Skip-gram Embeddings (t-SNE)",
                words_to_plot=interesting_words)

In [ ]:
# Check nearest neighbors in embedding space

def find_nearest(word, embeddings, word2idx, idx2word, top_k=5):
    """Find the top-k most similar words by cosine similarity."""
    if word not in word2idx:
        print(f"'{word}' not in vocabulary")
        return []
    word_vec = embeddings[word2idx[word]]
    # Compute cosine similarity against all words
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    norms = np.maximum(norms, 1e-10)  # avoid division by zero
    normed = embeddings / norms
    word_normed = word_vec / np.maximum(np.linalg.norm(word_vec), 1e-10)
    sims = normed @ word_normed
    # Get top-k (excluding the word itself)
    top_indices = sims.argsort()[::-1][1:top_k + 1]
    results = [(idx2word[i], sims[i]) for i in top_indices]
    return results


for word in ["king", "woman", "dog", "castle"]:
    neighbors = find_nearest(word, sg_embeddings, word2idx, idx2word)
    neighbor_str = ", ".join(f"{w} ({s:.3f})" for w, s in neighbors)
    print(f"Nearest to '{word}': {neighbor_str}")

---
## 3. Word2Vec -- CBOW

**Continuous Bag of Words (CBOW)** is the other Word2Vec variant. Where Skip-gram predicts context words from a center word, CBOW does the reverse: it predicts the center word from the average of the surrounding context word embeddings. In practice, CBOW tends to be faster to train and works slightly better for frequent words, while Skip-gram tends to produce better representations for rare words.

The CBOW architecture takes the embeddings of all context words within the window, averages them, and uses that averaged vector to predict the center word. The training objective is analogous to Skip-gram but with the roles reversed. We'll again use negative sampling for efficient training.

In [ ]:
# Create CBOW training data: (list_of_context_words, center_word)

def create_cbow_data(tokens, word2idx, window_size=2):
    """Generate (context_indices, center_index) pairs."""
    data = []
    for i in range(window_size, len(tokens) - window_size):
        context = []
        for j in range(i - window_size, i + window_size + 1):
            if j != i:
                context.append(word2idx[tokens[j]])
        center = word2idx[tokens[i]]
        data.append((context, center))
    return data


cbow_data = create_cbow_data(corpus_tokens, word2idx, window_size=WINDOW_SIZE)
print(f"Generated {len(cbow_data)} CBOW training examples")
print(f"\nExample: context words -> center word")
ctx, ctr = cbow_data[5]
print(f"  {[idx2word[c] for c in ctx]} -> '{idx2word[ctr]}'")

In [ ]:
class CBOWNegSampling(nn.Module):
    """Continuous Bag of Words model with negative sampling.

    Takes the average of context word embeddings and uses it
    to predict the center word.
    """

    def __init__(self, vocab_size: int, embedding_dim: int):
        super().__init__()
        self.embeddings = nn.Embedding(vocab_size, embedding_dim)
        self.output_embeddings = nn.Embedding(vocab_size, embedding_dim)

        initrange = 0.5 / embedding_dim
        self.embeddings.weight.data.uniform_(-initrange, initrange)
        self.output_embeddings.weight.data.uniform_(-initrange, initrange)

    def forward(self, context_ids, center_ids, negative_ids):
        """
        Args:
            context_ids:  (batch_size, 2*window) context word indices
            center_ids:   (batch_size,) center word indices
            negative_ids: (batch_size, num_negatives) negative sample indices

        Returns:
            Negative sampling loss (scalar)
        """
        # Average context embeddings: (batch, 2*window, dim) -> (batch, dim)
        context_vecs = self.embeddings(context_ids).mean(dim=1)

        # Center (positive) output embeddings: (batch, dim)
        center_vecs = self.output_embeddings(center_ids)

        # Negative output embeddings: (batch, num_neg, dim)
        neg_vecs = self.output_embeddings(negative_ids)

        # Positive loss
        pos_score = torch.sum(context_vecs * center_vecs, dim=1)
        pos_loss = torch.nn.functional.logsigmoid(pos_score)

        # Negative loss
        neg_score = torch.bmm(neg_vecs, context_vecs.unsqueeze(2)).squeeze(2)
        neg_loss = torch.nn.functional.logsigmoid(-neg_score).sum(dim=1)

        loss = -(pos_loss + neg_loss).mean()
        return loss

    def get_embeddings(self):
        """Return the input embeddings as a numpy array."""
        return self.embeddings.weight.data.cpu().numpy()


print("CBOWNegSampling model defined.")

In [ ]:
# Train the CBOW model

model_cbow = CBOWNegSampling(V, EMBEDDING_DIM).to(device)
optimizer_cbow = optim.Adam(model_cbow.parameters(), lr=LEARNING_RATE)

# Prepare tensors
context_size = 2 * WINDOW_SIZE
cbow_contexts = torch.tensor([c for c, _ in cbow_data], dtype=torch.long, device=device)
cbow_centers = torch.tensor([t for _, t in cbow_data], dtype=torch.long, device=device)

cbow_losses = []
num_examples = len(cbow_data)
num_batches_cbow = math.ceil(num_examples / BATCH_SIZE)

for epoch in range(NUM_EPOCHS):
    perm = torch.randperm(num_examples, device=device)
    ctx_shuf = cbow_contexts[perm]
    ctr_shuf = cbow_centers[perm]

    epoch_loss = 0.0
    for i in range(0, num_examples, BATCH_SIZE):
        batch_ctx = ctx_shuf[i:i + BATCH_SIZE]
        batch_ctr = ctr_shuf[i:i + BATCH_SIZE]
        batch_size = batch_ctr.size(0)

        neg_samples = torch.tensor(
            np.random.choice(V, size=(batch_size, NUM_NEGATIVES), p=noise_dist),
            dtype=torch.long, device=device
        )

        optimizer_cbow.zero_grad()
        loss = model_cbow(batch_ctx, batch_ctr, neg_samples)
        loss.backward()
        optimizer_cbow.step()

        epoch_loss += loss.item()

    avg_loss = epoch_loss / num_batches_cbow
    cbow_losses.append(avg_loss)
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch + 1:3d}/{NUM_EPOCHS} | Loss: {avg_loss:.4f}")

print("\nCBOW training complete!")

In [ ]:
# Compare Skip-gram vs CBOW training curves
plt.figure(figsize=(8, 4))
plt.plot(losses, label="Skip-gram", linewidth=2)
plt.plot(cbow_losses, label="CBOW", linewidth=2)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Skip-gram vs CBOW Training Loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Compare nearest neighbors from both models

cbow_embeddings = model_cbow.get_embeddings()

print("=" * 60)
print("Nearest Neighbors: Skip-gram vs CBOW")
print("=" * 60)
for word in ["king", "woman", "dog"]:
    sg_nn = find_nearest(word, sg_embeddings, word2idx, idx2word, top_k=3)
    cbow_nn = find_nearest(word, cbow_embeddings, word2idx, idx2word, top_k=3)
    print(f"\n'{word}':")
    print(f"  Skip-gram: {', '.join(f'{w} ({s:.3f})' for w, s in sg_nn)}")
    print(f"  CBOW:      {', '.join(f'{w} ({s:.3f})' for w, s in cbow_nn)}")

In [ ]:
# Visualize CBOW embeddings side by side with Skip-gram
plot_embeddings(cbow_embeddings, idx2word, title="CBOW Embeddings (t-SNE)",
                words_to_plot=interesting_words)

### Skip-gram vs CBOW: Key Differences

Both models learn from the distributional hypothesis, but they differ in important ways:

- **Skip-gram** predicts context from center word. It creates more training examples per word occurrence (one pair per context position), which gives rare words more representation. It tends to perform better on syntactic and semantic analogy tasks, especially for infrequent words.

- **CBOW** predicts center from context. It averages context vectors, making it faster to train. It tends to produce slightly better representations for frequent words because it smooths over multiple context signals.

On our small corpus the differences may be subtle, but on large-scale data (billions of tokens), Skip-gram generally produces higher-quality embeddings for downstream tasks, which is why it became the more popular variant.

---
## 4. GloVe

**GloVe** (Global Vectors for Word Representation), introduced by Pennington, Socher, and Manning (2014), takes a different approach from Word2Vec. Instead of training on individual (center, context) pairs through a neural network, GloVe first constructs a global **co-occurrence matrix** and then learns embeddings by factorizing this matrix with a weighted least-squares objective.

The key insight of GloVe is that the *ratio* of co-occurrence probabilities encodes meaning. If word $k$ is related to "ice" but not "steam," then $P(k|\text{ice}) / P(k|\text{steam})$ will be large. GloVe's objective function directly captures this by requiring:

$$\mathbf{w}_i^\top \mathbf{w}_j + b_i + b_j = \log(X_{ij})$$

where $X_{ij}$ is the co-occurrence count. The loss is weighted so that very frequent and very rare co-occurrences don't dominate:

$$J = \sum_{i,j=1}^{V} f(X_{ij}) \left( \mathbf{w}_i^\top \tilde{\mathbf{w}}_j + b_i + \tilde{b}_j - \log X_{ij} \right)^2$$

where $f(x) = \min(1, (x/x_{\max})^\alpha)$ with $\alpha = 0.75$ typically.

In [ ]:
# Build co-occurrence matrix from corpus

def build_cooccurrence_matrix(tokens, word2idx, window_size=3):
    """Build a symmetric co-occurrence matrix with distance weighting.

    Words closer to the center word receive higher co-occurrence counts
    (weight = 1/distance), following the GloVe paper.
    """
    vocab_size = len(word2idx)
    cooccurrence = defaultdict(float)

    for i, token in enumerate(tokens):
        token_idx = word2idx[token]
        for j in range(max(0, i - window_size), min(len(tokens), i + window_size + 1)):
            if i == j:
                continue
            context_idx = word2idx[tokens[j]]
            distance = abs(i - j)
            # Weight by inverse distance (closer words co-occur more strongly)
            cooccurrence[(token_idx, context_idx)] += 1.0 / distance

    return cooccurrence


cooccurrence = build_cooccurrence_matrix(corpus_tokens, word2idx, window_size=WINDOW_SIZE)
print(f"Non-zero co-occurrence entries: {len(cooccurrence)}")

# Show some co-occurrence values
print("\nSample co-occurrences:")
sample_pairs = [("king", "queen"), ("king", "kingdom"), ("king", "fish"), ("dog", "cat")]
for w1, w2 in sample_pairs:
    val = cooccurrence.get((word2idx[w1], word2idx[w2]), 0)
    print(f"  X('{w1}', '{w2}') = {val:.2f}")

In [ ]:
class GloVeModel(nn.Module):
    """GloVe: Global Vectors for Word Representation.

    Learns two sets of embeddings (W and W_tilde) plus bias terms
    by minimizing weighted least squares on the log co-occurrence matrix.
    """

    def __init__(self, vocab_size: int, embedding_dim: int, x_max: float = 100.0, alpha: float = 0.75):
        super().__init__()
        self.x_max = x_max
        self.alpha = alpha

        # Two embedding matrices (center and context)
        self.w_embeddings = nn.Embedding(vocab_size, embedding_dim)
        self.w_tilde_embeddings = nn.Embedding(vocab_size, embedding_dim)

        # Bias terms
        self.w_bias = nn.Embedding(vocab_size, 1)
        self.w_tilde_bias = nn.Embedding(vocab_size, 1)

        # Initialize
        initrange = 0.5 / embedding_dim
        self.w_embeddings.weight.data.uniform_(-initrange, initrange)
        self.w_tilde_embeddings.weight.data.uniform_(-initrange, initrange)
        self.w_bias.weight.data.zero_()
        self.w_tilde_bias.weight.data.zero_()

    def weighting_function(self, x):
        """f(x) = min(1, (x/x_max)^alpha)"""
        return torch.clamp((x / self.x_max) ** self.alpha, max=1.0)

    def forward(self, i_indices, j_indices, co_values):
        """
        Args:
            i_indices:  (batch,) word i indices
            j_indices:  (batch,) word j indices
            co_values:  (batch,) X_ij co-occurrence values

        Returns:
            Weighted least squares loss
        """
        w_i = self.w_embeddings(i_indices)            # (batch, dim)
        w_j = self.w_tilde_embeddings(j_indices)      # (batch, dim)
        b_i = self.w_bias(i_indices).squeeze(1)       # (batch,)
        b_j = self.w_tilde_bias(j_indices).squeeze(1) # (batch,)

        # Predicted log co-occurrence
        dot_product = torch.sum(w_i * w_j, dim=1)     # (batch,)
        prediction = dot_product + b_i + b_j

        # Target: log(X_ij)
        log_cooccurrence = torch.log(co_values + 1e-10)

        # Weighted squared error
        weights = self.weighting_function(co_values)
        loss = weights * (prediction - log_cooccurrence) ** 2

        return loss.mean()

    def get_embeddings(self):
        """Return combined embeddings (W + W_tilde) as in the GloVe paper."""
        w = self.w_embeddings.weight.data.cpu().numpy()
        w_tilde = self.w_tilde_embeddings.weight.data.cpu().numpy()
        return w + w_tilde


print("GloVeModel defined.")

In [ ]:
# Train GloVe

model_glove = GloVeModel(V, EMBEDDING_DIM).to(device)
optimizer_glove = optim.Adam(model_glove.parameters(), lr=0.05)

# Prepare co-occurrence data as tensors
cooc_i = []
cooc_j = []
cooc_val = []
for (i, j), val in cooccurrence.items():
    cooc_i.append(i)
    cooc_j.append(j)
    cooc_val.append(val)

cooc_i = torch.tensor(cooc_i, dtype=torch.long, device=device)
cooc_j = torch.tensor(cooc_j, dtype=torch.long, device=device)
cooc_val = torch.tensor(cooc_val, dtype=torch.float32, device=device)

num_cooc = len(cooc_i)
GLOVE_EPOCHS = 100
GLOVE_BATCH = 512
glove_losses = []

for epoch in range(GLOVE_EPOCHS):
    perm = torch.randperm(num_cooc, device=device)
    i_shuf = cooc_i[perm]
    j_shuf = cooc_j[perm]
    v_shuf = cooc_val[perm]

    epoch_loss = 0.0
    n_batches = 0
    for b in range(0, num_cooc, GLOVE_BATCH):
        batch_i = i_shuf[b:b + GLOVE_BATCH]
        batch_j = j_shuf[b:b + GLOVE_BATCH]
        batch_v = v_shuf[b:b + GLOVE_BATCH]

        optimizer_glove.zero_grad()
        loss = model_glove(batch_i, batch_j, batch_v)
        loss.backward()
        optimizer_glove.step()

        epoch_loss += loss.item()
        n_batches += 1

    avg_loss = epoch_loss / n_batches
    glove_losses.append(avg_loss)
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch + 1:3d}/{GLOVE_EPOCHS} | Loss: {avg_loss:.4f}")

print("\nGloVe training complete!")

In [ ]:
# Plot GloVe training curve
plt.figure(figsize=(8, 4))
plt.plot(glove_losses, linewidth=2, color="green")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("GloVe Training Loss")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Compare GloVe vs Word2Vec (Skip-gram) nearest neighbors

glove_embeddings = model_glove.get_embeddings()

print("=" * 60)
print("Nearest Neighbors: Skip-gram vs GloVe")
print("=" * 60)
for word in ["king", "woman", "dog", "castle"]:
    sg_nn = find_nearest(word, sg_embeddings, word2idx, idx2word, top_k=3)
    gl_nn = find_nearest(word, glove_embeddings, word2idx, idx2word, top_k=3)
    print(f"\n'{word}':")
    print(f"  Skip-gram: {', '.join(f'{w} ({s:.3f})' for w, s in sg_nn)}")
    print(f"  GloVe:     {', '.join(f'{w} ({s:.3f})' for w, s in gl_nn)}")

In [ ]:
# Visualize GloVe embeddings
plot_embeddings(glove_embeddings, idx2word, title="GloVe Embeddings (t-SNE)",
                words_to_plot=interesting_words)

### GloVe vs Word2Vec: Key Differences

While both methods produce high-quality word embeddings, they differ fundamentally in how they learn:

- **Word2Vec** is a *predictive* model that learns embeddings through a neural network training loop on individual word pairs. It processes the corpus one window at a time, making it a local, online method.

- **GloVe** is a *count-based* model that first aggregates global co-occurrence statistics and then factorizes the resulting matrix. It leverages global corpus statistics directly, which the authors argue gives it a theoretical advantage.

In practice, on large corpora with well-tuned hyperparameters, the two methods produce embeddings of comparable quality. The choice often comes down to implementation convenience and specific task requirements. GloVe's explicit use of the co-occurrence matrix can make it easier to analyze what the model has learned.

---
## 5. Evaluating Embeddings

How do we know if our embeddings are any good? There are several standard evaluation methods:

1. **Word analogies**: The classic "king - man + woman = queen" test. Good embeddings should capture linear relationships between word pairs that reflect semantic or syntactic regularities. If the vector arithmetic works, the embedding space has learned meaningful structure.

2. **Similarity benchmarks**: Compare the model's similarity rankings to human judgments. Datasets like WordSim-353 and SimLex-999 provide human-rated word pair similarities that we can correlate with cosine similarities in embedding space.

3. **Clustering and visualization**: Words with related meanings should cluster together. We can use t-SNE or UMAP to visualize this and qualitatively assess whether the space captures the semantic relationships we expect.

Let's implement and run all three evaluation approaches on our trained embeddings.

In [ ]:
# Word Analogy Solver

def solve_analogy(a, b, c, embeddings, word2idx, idx2word, top_k=3):
    """Solve: a is to b as c is to ?

    Uses vector arithmetic: result = b - a + c
    Then finds the nearest word to the result vector.
    """
    if any(w not in word2idx for w in [a, b, c]):
        missing = [w for w in [a, b, c] if w not in word2idx]
        return f"Words not in vocabulary: {missing}"

    vec_a = embeddings[word2idx[a]]
    vec_b = embeddings[word2idx[b]]
    vec_c = embeddings[word2idx[c]]

    # Analogy vector: b - a + c
    result_vec = vec_b - vec_a + vec_c

    # Find nearest words (excluding the input words)
    exclude = {word2idx[a], word2idx[b], word2idx[c]}
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    norms = np.maximum(norms, 1e-10)
    normed = embeddings / norms
    result_normed = result_vec / np.maximum(np.linalg.norm(result_vec), 1e-10)
    sims = normed @ result_normed

    # Zero out excluded words
    for idx in exclude:
        sims[idx] = -1.0

    top_indices = sims.argsort()[::-1][:top_k]
    return [(idx2word[i], sims[i]) for i in top_indices]


# Test analogies on our embeddings
analogy_tests = [
    ("king", "queen", "man", "woman"),     # king:queen :: man:?
    ("king", "queen", "prince", "princess"),# king:queen :: prince:?
    ("man", "woman", "boy", "girl"),        # man:woman :: boy:?
    ("king", "prince", "queen", "princess"),# king:prince :: queen:?
]

for embeddings_name, embeddings in [("Skip-gram", sg_embeddings), ("GloVe", glove_embeddings)]:
    print(f"\n{'=' * 50}")
    print(f"Analogies with {embeddings_name} embeddings:")
    print(f"{'=' * 50}")
    for a, b, c, expected in analogy_tests:
        results = solve_analogy(a, b, c, embeddings, word2idx, idx2word)
        top_word = results[0][0] if isinstance(results, list) else "N/A"
        match = " ✓" if top_word == expected else f" (expected: {expected})"
        print(f"  {a}:{b} :: {c}:? -> {top_word}{match}")
        if isinstance(results, list):
            for w, s in results:
                print(f"    {w}: {s:.3f}")

In [ ]:
# Similarity evaluation: compute pairwise similarities and compare to intuition

def pairwise_similarities(words, embeddings, word2idx):
    """Compute cosine similarity matrix for a list of words."""
    indices = [word2idx[w] for w in words]
    vecs = embeddings[indices]
    norms = np.linalg.norm(vecs, axis=1, keepdims=True)
    norms = np.maximum(norms, 1e-10)
    normed = vecs / norms
    sim_matrix = normed @ normed.T
    return sim_matrix


eval_words = ["king", "queen", "prince", "princess", "man", "woman",
              "boy", "girl", "dog", "cat", "fish"]

sim_matrix = pairwise_similarities(eval_words, sg_embeddings, word2idx)

plt.figure(figsize=(9, 7))
plt.imshow(sim_matrix, cmap="RdYlBu_r", vmin=-1, vmax=1)
plt.xticks(range(len(eval_words)), eval_words, rotation=45, ha="right")
plt.yticks(range(len(eval_words)), eval_words)
plt.colorbar(label="Cosine Similarity")
plt.title("Skip-gram: Pairwise Cosine Similarities")
plt.tight_layout()
plt.show()

In [ ]:
# Clustering visualization with color-coded semantic groups

def plot_clusters(embeddings, word2idx, idx2word, word_groups, title):
    """Visualize word embeddings with color-coded semantic groups."""
    all_words = []
    all_colors = []
    color_map = plt.cm.Set1

    for i, (group_name, words) in enumerate(word_groups.items()):
        for w in words:
            if w in word2idx:
                all_words.append(w)
                all_colors.append(color_map(i / len(word_groups)))

    indices = [word2idx[w] for w in all_words]
    vecs = embeddings[indices]

    perplexity = min(5, len(indices) - 1)
    tsne = TSNE(n_components=2, random_state=42, perplexity=perplexity)
    vecs_2d = tsne.fit_transform(vecs)

    plt.figure(figsize=(10, 8))
    for i, (group_name, words) in enumerate(word_groups.items()):
        mask = [all_words[j] in words for j in range(len(all_words))]
        mask_indices = [j for j, m in enumerate(mask) if m]
        if mask_indices:
            plt.scatter(vecs_2d[mask_indices, 0], vecs_2d[mask_indices, 1],
                        c=[all_colors[mask_indices[0]]], label=group_name,
                        s=100, alpha=0.7)

    for i, word in enumerate(all_words):
        plt.annotate(word, (vecs_2d[i, 0], vecs_2d[i, 1]),
                     fontsize=10, fontweight="bold",
                     xytext=(5, 5), textcoords="offset points")

    plt.legend(fontsize=11)
    plt.title(title, fontsize=14)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


word_groups = {
    "Royalty": ["king", "queen", "prince", "princess", "royal", "kingdom"],
    "People": ["man", "woman", "boy", "girl"],
    "Animals": ["dog", "cat", "fish", "bird"],
    "Places": ["castle", "garden", "park", "city"],
}

plot_clusters(sg_embeddings, word2idx, idx2word, word_groups,
              "Skip-gram Embeddings: Semantic Clusters")

### Evaluation Summary

Even on our small corpus, we can see that the embedding models have learned meaningful structure:

- **Analogies** may partially work -- with a small corpus, the linear relationships are noisier than with large-scale training, but the direction is correct.
- **Similarity matrices** show that semantically related words (king/queen, dog/cat) tend to have higher cosine similarity.
- **Cluster visualizations** show that words group by semantic category (royalty, animals, people, places).

On real-world corpora with billions of tokens and larger embedding dimensions, these patterns become much cleaner and the analogy task works remarkably well.

---
## Summary

In this notebook, we covered the **foundations of word embeddings**:

1. **One-hot encoding** is sparse, high-dimensional, and captures no semantic relationships.
2. **Word2Vec Skip-gram** learns embeddings by predicting context words from a center word using negative sampling.
3. **Word2Vec CBOW** predicts the center word from averaged context embeddings -- faster but less effective for rare words.
4. **GloVe** factorizes a global co-occurrence matrix, combining count-based and predictive approaches.
5. **Evaluation** via word analogies, similarity benchmarks, and clustering visualization.

All three methods produce **static embeddings** -- each word gets exactly one vector regardless of context. In the next notebook, we'll explore the limitations of this approach and how **contextual embeddings** (ELMo, BERT, etc.) address them.

---
**Next:** `advanced.ipynb` -- From static to contextual embeddings, sentence embeddings, and modern embedding models.